# Flow-Aware Temporal Pattern Mining for Multi-Stage Network Intrusion Detection (NIDS)

**M.Tech Major Project** — CIC-IDS2017 Dataset

This notebook implements a 13-stage multi-stage NIDS pipeline that fuses:
- Flow-level tabular classifiers (Random Forest, XGBoost)
- Session-level sequence modelling (BiLSTM over behavioral event tokens)
- Unsupervised frequent-pattern mining (FP-Growth + PrefixSpan)
- A learned attack-state transition graph with temporal consistency scoring
- An adaptive evidence-fusion + risk meta-learning layer
- Streaming (windowed) evaluation and SHAP/graph-based explainability

All stages import their logic from `src/nids/*.py` — this notebook is the
orchestration / experiment-tracking layer, not where the algorithms live.

**Chronological split (no shuffling, to prevent temporal leakage):**
| Split | Days | Files |
|---|---|---|
| Train | 1–2 | Monday, Tuesday |
| Validation | 3 | Wednesday |
| Test | 4–7 | Thursday (AM/PM), Friday (AM/PM) |


## Cell 0 — Environment Setup

In [ ]:
# !pip install -q mlxtend shap xgboost imbalanced-learn networkx seaborn tensorflow

import sys
import warnings
from pathlib import Path

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import networkx as nx

from config import *

matplotlib.rcParams.update({
    "figure.dpi": FIGURE_DPI,
    "savefig.dpi": FIGURE_DPI,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
PALETTE = sns.color_palette(PALETTE_NAME, K)
sns.set_style("whitegrid")

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)
print(f"Seed set to {SEED}. Target classes K={K}. Figures -> {FIG_DIR.resolve()}")


## Stage 1 — Data Acquisition & Chronological Partition

Loads all 7 CIC-IDS2017 day files and assembles the chronological
Train (Days 1-2) / Validation (Day 3) / Test (Days 4-7) split described in
the project methodology. Column-name whitespace, the `" Label"` quirk, and
duplicate embedded header rows are cleaned inside `data_loading.load_day`.


In [ ]:
from src.nids.data_loading import load_all_splits

df_train_raw, df_val_raw, df_test_raw = load_all_splits(DATASET_DIR)

print("\n=== Split sizes ===")
print(f"Train : {df_train_raw.shape[0]:>10,} rows  (Days 1-2)")
print(f"Val   : {df_val_raw.shape[0]:>10,} rows  (Day 3)")
print(f"Test  : {df_test_raw.shape[0]:>10,} rows  (Days 4-7)")

print("\n=== Class counts (Train) ===")
print(df_train_raw["Label"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, df) in zip(axes, [("Train", df_train_raw), ("Validation", df_val_raw), ("Test", df_test_raw)]):
    counts = df["Label"].value_counts()
    ax.barh(counts.index, counts.values, color=PALETTE[: len(counts)])
    ax.set_xscale("log")
    ax.set_title(f"{name} (n={len(df):,})")
    ax.set_xlabel("Flow count (log scale)")
fig.suptitle("Stage 1 — Class Distribution per Chronological Split", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage1_class_distribution.png", bbox_inches="tight")
plt.show()


## Stage 2 — Preprocessing & Imbalance Handling

Cleans inf/NaN, clips the 99th-percentile outliers (fit on train only),
MinMax-scales flow-statistic features, encodes labels, computes
inverse-frequency class weights, and builds the SMOTE+ENN-augmented
**Branch B** used by the tabular (RF/XGBoost) classifiers. **Branch A**
(raw, class-weighted) is retained for the BiLSTM sequence branch.


In [ ]:
from src.nids.preprocessing import (
    preprocess, encode_labels, compute_class_weights, smote_knn_augment, get_feature_cols,
)

df_train, df_val, df_test, scaler, FEAT_COLS = preprocess(df_train_raw, df_val_raw, df_test_raw)
le, CLASSES, K = encode_labels(df_train, df_val, df_test)
CLASS_WEIGHTS = compute_class_weights(df_train, le, K)

X_train_A = df_train[FEAT_COLS].values
y_train_A = df_train["LabelID"].values
X_val = df_val[FEAT_COLS].values
y_val = df_val["LabelID"].values
X_test = df_test[FEAT_COLS].values
y_test = df_test["LabelID"].values

class_counts = df_train["LabelID"].value_counts().to_dict()
X_train_B, y_train_B = smote_knn_augment(X_train_A, y_train_A, class_counts, K, SEED)

print(f"Branch A (raw, class-weighted)   : X={X_train_A.shape}, y={y_train_A.shape}")
print(f"Branch B (SMOTE+ENN augmented)   : X={X_train_B.shape}, y={y_train_B.shape}")
print(f"\n{len(FEAT_COLS)} ML feature columns (identifier/5-tuple columns excluded).")

weight_table = pd.DataFrame({
    "class": CLASSES,
    "train_count": [class_counts.get(i, 0) for i in range(K)],
    "class_weight": [CLASS_WEIGHTS[i] for i in range(K)],
}).sort_values("train_count")
weight_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(weight_table["class"], weight_table["train_count"], color=PALETTE[0])
axes[0].set_xscale("log")
axes[0].set_title("Train class counts (log scale)")

axes[1].barh(weight_table["class"], weight_table["class_weight"], color=PALETTE[1])
axes[1].set_title("Inverse-frequency class weights")
fig.suptitle("Stage 2 — Imbalance Handling", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage2_imbalance.png", bbox_inches="tight")
plt.show()


## Stage 3 — Feature Engineering

Ranks features by mutual information with the class label, then assigns
them into four semantically grounded groups (general/top-MI, handshake &
header, duration/IAT, behavioral flag counts) used to build per-model
feature views: RF -> A∪B, XGBoost -> B∪D, BiLSTM -> token sequence.


In [ ]:
from src.nids.feature_engineering import compute_mi_scores, assign_feature_groups, get_model_features

mi_scores = compute_mi_scores(df_train[FEAT_COLS], y_train_A, FEAT_COLS, SEED)
GROUP_A, GROUP_B, GROUP_C, GROUP_D = assign_feature_groups(FEAT_COLS, mi_scores)
RF_FEATURES, LSTM_FEATURES, XGB_FEATURES = get_model_features(GROUP_A, GROUP_B, GROUP_C, GROUP_D)

print(f"GROUP_A (general/top-MI)      : {len(GROUP_A)} features")
print(f"GROUP_B (handshake/header)    : {len(GROUP_B)} features")
print(f"GROUP_C (duration/IAT)        : {len(GROUP_C)} features")
print(f"GROUP_D (behavioral flag cnt) : {len(GROUP_D)} features")
print(f"\nRF_FEATURES  (A∪B) : {len(RF_FEATURES)} features")
print(f"XGB_FEATURES (B∪D) : {len(XGB_FEATURES)} features")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top30 = mi_scores.head(30)
axes[0].barh(top30.index[::-1], top30.values[::-1], color=PALETTE[2])
axes[0].set_title("Top-30 Features by Mutual Information")
axes[0].set_xlabel("MI score")

group_sizes = [len(GROUP_A), len(GROUP_B), len(GROUP_C), len(GROUP_D)]
group_names = ["GROUP_A", "GROUP_B", "GROUP_C", "GROUP_D"]
axes[1].pie(group_sizes, labels=group_names, autopct="%1.0f%%", colors=PALETTE[3:7],
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Feature Group Sizes")

fig.suptitle("Stage 3 — Feature Engineering", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage3_feature_groups.png", bbox_inches="tight")
plt.show()


## Stage 4 — Bidirectional Session Reconstruction

Groups uni-directional flow records into bidirectional network sessions
keyed on a symmetric 5-tuple (min/max IP, protocol, min/max port), with a
`SESSION_TIMEOUT`-second idle gap splitting a re-used 5-tuple into a new
session.


In [ ]:
from src.nids.sessions import reconstruct_sessions, get_session_stats

df_train_sess = reconstruct_sessions(df_train, timeout=SESSION_TIMEOUT)
df_val_sess = reconstruct_sessions(df_val, timeout=SESSION_TIMEOUT)
df_test_sess = reconstruct_sessions(df_test, timeout=SESSION_TIMEOUT)

stats_train = get_session_stats(df_train_sess)
print(f"Train sessions : {df_train_sess['SessionID'].nunique():,} (from {len(df_train_sess):,} flows)")
print(f"Val sessions   : {df_val_sess['SessionID'].nunique():,} (from {len(df_val_sess):,} flows)")
print(f"Test sessions  : {df_test_sess['SessionID'].nunique():,} (from {len(df_test_sess):,} flows)")
print("\nSession size stats (train):")
print(stats_train["n_flows"].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(stats_train["n_flows"], bins=40, color=PALETTE[4])
axes[0].set_yscale("log")
axes[0].set_xlabel("Flows per session")
axes[0].set_ylabel("Count (log scale)")
axes[0].set_title("Session Size Distribution (Train)")

sess_per_class = stats_train.groupby("label")["session_id"].count().sort_values()
axes[1].barh(sess_per_class.index, sess_per_class.values, color=PALETTE[5])
axes[1].set_title("Sessions per Class (Train)")

fig.suptitle("Stage 4 — Bidirectional Session Reconstruction", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage4_sessions.png", bbox_inches="tight")
plt.show()


## Stage 5 — Behavioral Event Encoding

Maps every flow row to one of 10 coarse behavioral tokens (handshake state,
scan/auth/DoS indicators, data volume), then builds padded/truncated
per-session token sequences for the BiLSTM branch.


In [ ]:
from src.nids.events import encode_sessions, sessions_to_arrays, compute_sample_weights_lstm

df_train_sess, SESSIONS_TRAIN = encode_sessions(df_train_sess)
df_val_sess, SESSIONS_VAL = encode_sessions(df_val_sess)
df_test_sess, SESSIONS_TEST = encode_sessions(df_test_sess)

X_lstm_train, y_lstm_train = sessions_to_arrays(SESSIONS_TRAIN, le, max_len=MAX_SEQ_LEN)
X_lstm_val, y_lstm_val = sessions_to_arrays(SESSIONS_VAL, le, max_len=MAX_SEQ_LEN)
X_lstm_test, y_lstm_test = sessions_to_arrays(SESSIONS_TEST, le, max_len=MAX_SEQ_LEN)

sample_weights_lstm = compute_sample_weights_lstm(y_lstm_train, CLASS_WEIGHTS)

# Session-level ground truth (reused by fusion/risk/streaming stages).
val_session_ids = list(SESSIONS_VAL.keys())
test_session_ids = list(SESSIONS_TEST.keys())
y_val_sess = y_lstm_val
y_test_sess = y_lstm_test

token_counts = df_train_sess["Token"].value_counts()
print("Token frequency (train flows):")
print(token_counts)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(token_counts.index[::-1], token_counts.values[::-1], color=PALETTE[6])
axes[0].set_xscale("log")
axes[0].set_title("Behavioral Token Frequency (Train)")

def _pick_example_session(sessions_dict):
    for min_len in (3, 2, 1):
        for sid, s in sessions_dict.items():
            if s["label"] != "BENIGN" and len(s["tokens"]) >= min_len:
                return sid
    return next(iter(sessions_dict))

example_sid = _pick_example_session(SESSIONS_TRAIN)
example_tokens = SESSIONS_TRAIN[example_sid]["tokens"]
token_idx = [VOCAB.index(t) for t in example_tokens]
axes[1].step(range(len(token_idx)), token_idx, where="mid", color=PALETTE[7], marker="o")
axes[1].set_yticks(range(len(VOCAB)))
axes[1].set_yticklabels(VOCAB, fontsize=8)
axes[1].set_xlabel("Event index in session")
axes[1].set_title(f"Example Attack Session ({SESSIONS_TRAIN[example_sid]['label']})")

fig.suptitle("Stage 5 — Behavioral Event Encoding", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage5_token_encoding.png", bbox_inches="tight")
plt.show()


## Stage 6 — Parallel Classifiers

### 6a. Random Forest (GROUP_A ∪ GROUP_B tabular features, Branch B)


In [ ]:
from src.nids.models import train_rf, evaluate_classifier, align_proba_to_k
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

rf_col_idx = [FEAT_COLS.index(f) for f in RF_FEATURES]
X_rf_train_B = X_train_B[:, rf_col_idx]
X_rf_val = X_val[:, rf_col_idx]
X_rf_test = X_test[:, rf_col_idx]

rf = train_rf(X_rf_train_B, y_train_B, CLASS_WEIGHTS, SEED)
# Re-index onto the full K-class space: SMOTE+ENN can fully remove an
# extremely rare class (e.g. Heartbleed, N=11) from the training fold,
# leaving rf.classes_ / predict_proba narrower than K.
P_A_val = align_proba_to_k(rf.predict_proba(X_rf_val), rf.classes_, K)
P_A_test = align_proba_to_k(rf.predict_proba(X_rf_test), rf.classes_, K)

rf_metrics = evaluate_classifier(rf, X_rf_test, y_test, "Random Forest", CLASSES)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(rf_metrics["confusion_matrix"], annot=False, cmap="Blues", ax=axes[0],
            xticklabels=CLASSES, yticklabels=CLASSES, cbar=True)
axes[0].set_title("RF Confusion Matrix (Test)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
plt.setp(axes[0].get_xticklabels(), rotation=90)

f1_per_class = [rf_metrics["classification_report"][c]["f1-score"] for c in CLASSES]
axes[1].barh(CLASSES, f1_per_class, color=PALETTE[: len(CLASSES)])
axes[1].set_title("RF Per-Class F1 (Test)")
axes[1].set_xlim(0, 1)

fig.suptitle("Stage 6a — Random Forest Results", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage6a_rf_results.png", bbox_inches="tight")
plt.show()


In [ ]:
y_test_bin = label_binarize(y_test, classes=list(range(K)))
fig, ax = plt.subplots(figsize=(9, 8))
for i in range(K):
    if y_test_bin[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], P_A_test[:, i])
    ax.plot(fpr, tpr, label=f"{CLASSES[i]} (AUC={auc(fpr, tpr):.2f})", color=PALETTE[i])
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Stage 6a — RF ROC Curves (One-vs-Rest, Test)")
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "stage6a_rf_roc.png", bbox_inches="tight")
plt.show()


### 6b. BiLSTM (behavioral token sequence, session-level)

In [ ]:
from src.nids.models import build_lstm, train_lstm

lstm_model = build_lstm(len(VOCAB), LSTM_EMBED_DIM, LSTM_UNITS, LSTM_DROPOUT, K, LSTM_LR)
lstm_model.summary()

history = train_lstm(
    lstm_model, X_lstm_train, y_lstm_train, sample_weights_lstm,
    X_lstm_val, y_lstm_val, LSTM_EPOCHS, LSTM_BATCH_SIZE, LSTM_PATIENCE,
)

P_B_val = lstm_model.predict(X_lstm_val, verbose=0)
P_B_test = lstm_model.predict(X_lstm_test, verbose=0)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(history.history["loss"], label="train", color=PALETTE[0])
axes[0].plot(history.history["val_loss"], label="val", color=PALETTE[1])
axes[0].set_title("BiLSTM Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train", color=PALETTE[0])
axes[1].plot(history.history["val_accuracy"], label="val", color=PALETTE[1])
axes[1].set_title("BiLSTM Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

fig.suptitle("Stage 6b — BiLSTM Training Curves", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage6b_lstm_training.png", bbox_inches="tight")
plt.show()


### 6c. XGBoost (GROUP_B ∪ GROUP_D tabular features, Branch B)

In [ ]:
from src.nids.models import train_xgboost

xgb_col_idx = [FEAT_COLS.index(f) for f in XGB_FEATURES]
X_xgb_train_B = X_train_B[:, xgb_col_idx]
X_xgb_val = X_val[:, xgb_col_idx]
X_xgb_test = X_test[:, xgb_col_idx]

xgb_clf = train_xgboost(X_xgb_train_B, y_train_B, CLASS_WEIGHTS, X_xgb_val, y_val, K, SEED)
P_C_val = align_proba_to_k(xgb_clf.predict_proba(X_xgb_val), xgb_clf.classes_, K)
P_C_test = align_proba_to_k(xgb_clf.predict_proba(X_xgb_test), xgb_clf.classes_, K)

xgb_metrics = evaluate_classifier(xgb_clf, X_xgb_test, y_test, "XGBoost", CLASSES)

lstm_pred_test = np.argmax(P_B_test, axis=1)
from sklearn.metrics import f1_score, accuracy_score
lstm_macro_f1 = f1_score(y_test_sess, lstm_pred_test, average="macro", zero_division=0)
lstm_acc = accuracy_score(y_test_sess, lstm_pred_test)

comparison = pd.DataFrame([
    {"model": "Random Forest", "features": "GROUP_A ∪ GROUP_B", "macro_f1": rf_metrics["macro_f1"], "accuracy": rf_metrics["accuracy"]},
    {"model": "BiLSTM", "features": "token sequence", "macro_f1": lstm_macro_f1, "accuracy": lstm_acc},
    {"model": "XGBoost", "features": "GROUP_B ∪ GROUP_D", "macro_f1": xgb_metrics["macro_f1"], "accuracy": xgb_metrics["accuracy"]},
])
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width / 2, comparison["macro_f1"], width, label="Macro-F1", color=PALETTE[0])
ax.bar(x + width / 2, comparison["accuracy"], width, label="Accuracy", color=PALETTE[1])
ax.set_xticks(x); ax.set_xticklabels(comparison["model"])
ax.set_ylim(0, 1); ax.legend()
ax.set_title("Stage 6 — Classifier Comparison (Test)")
fig.tight_layout()
fig.savefig(FIG_DIR / "stage6_comparison.png", bbox_inches="tight")
plt.show()


## Stage 7 — Adaptive Evidence Fusion

Pools flow-level RF/XGBoost probabilities to session level (mean-pooling
within each `SessionID`), then grid-searches the fusion weights
`(w_A, w_B, w_C, w_S, w_T)` on the validation set to maximise macro-F1.
Real `SP_t`/`TC_t` are not available until Stages 8-10, so calibration uses
neutral placeholders (0.5) — the weights are refined again in Stage 11 once
the real structural signals are computed.


In [ ]:
from src.nids.fusion import flow_probs_to_session, fuse, calibrate_fusion_weights

P_A_val_sess = flow_probs_to_session(df_val_sess, P_A_val, val_session_ids)
P_C_val_sess = flow_probs_to_session(df_val_sess, P_C_val, val_session_ids)
P_A_test_sess = flow_probs_to_session(df_test_sess, P_A_test, test_session_ids)
P_C_test_sess = flow_probs_to_session(df_test_sess, P_C_test, test_session_ids)

SP_t_placeholder = np.full(len(val_session_ids), 0.5)
TC_t_placeholder = np.full(len(val_session_ids), 0.5)

best_weights = calibrate_fusion_weights(
    P_A_val_sess, P_B_val, P_C_val_sess, SP_t_placeholder, TC_t_placeholder, y_val_sess, FUSION_GRID,
)
W_A, W_B, W_C, W_S, W_T = best_weights
print(f"Best weights: w_A={W_A}, w_B={W_B}, w_C={W_C}, w_S={W_S}, w_T={W_T}")
print("Weights FROZEN for downstream stages (re-calibrated in Stage 11 with real SP_t/TC_t).")

R_test_placeholder = fuse(
    P_A_test_sess, P_B_test, P_C_test_sess,
    np.full(len(test_session_ids), 0.5), np.full(len(test_session_ids), 0.5), *best_weights,
)
sample_idx = 0
sample_probs = pd.Series(R_test_placeholder[sample_idx], index=CLASSES).sort_values(ascending=False)
print(f"\nSample fused probability table (session {test_session_ids[sample_idx]}):")
print(sample_probs)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].pie([W_A, W_B, W_C, W_S, W_T], labels=["w_A (RF)", "w_B (BiLSTM)", "w_C (XGB)", "w_S (SP_t)", "w_T (TC_t)"],
            autopct="%1.0f%%", colors=PALETTE[:5], wedgeprops={"edgecolor": "white"})
axes[0].set_title("Calibrated Fusion Weights")

axes[1].hist(R_test_placeholder.max(axis=1), bins=30, color=PALETTE[2])
axes[1].set_xlabel("max_c R_t[c] (top-class confidence)")
axes[1].set_title("Fused Probability Distribution (Test, placeholder SP/TC)")

fig.suptitle("Stage 7 — Adaptive Evidence Fusion", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage7_fusion.png", bbox_inches="tight")
plt.show()


## Stage 8 — FP-Growth + PrefixSpan Pattern Mining

Mines frequent unordered token itemsets (FP-Growth) and gap-constrained
ordered token subsequences (PrefixSpan-style) from real (non-SMOTE)
training sessions, then scores every held-out session by how strongly it
matches the mined attack-behavior vocabulary (`SP_t`).


In [ ]:
from src.nids.pattern_mining import build_transaction_db, run_fpgrowth, run_prefixspan, compute_sp_scores_batch

te, df_trans = build_transaction_db(SESSIONS_TRAIN)
fp_patterns = run_fpgrowth(df_trans, FP_GROWTH_MIN_SUPPORT)
ps_patterns = run_prefixspan(SESSIONS_TRAIN, PREFIXSPAN_MIN_SUPPORT, PREFIXSPAN_MAX_GAP)

SP_t_val = compute_sp_scores_batch(SESSIONS_VAL, fp_patterns, ps_patterns)
SP_t_test = compute_sp_scores_batch(SESSIONS_TEST, fp_patterns, ps_patterns)
SP_t_val_arr = np.array([SP_t_val[s] for s in val_session_ids])
SP_t_test_arr = np.array([SP_t_test[s] for s in test_session_ids])

print(f"FP-Growth patterns mined : {len(fp_patterns)}")
print(f"PrefixSpan patterns mined: {len(ps_patterns)}")
print("\nTop-10 FP-Growth itemsets:")
print(fp_patterns.head(10))
print("\nTop-10 sequential patterns:")
for pattern, support in ps_patterns[:10]:
    print(f"  {' -> '.join(pattern)}  (support={support:.3f})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

top_fp = fp_patterns.head(15).copy()
top_fp["label"] = top_fp["itemsets"].apply(lambda s: ", ".join(sorted(s)))
axes[0].barh(top_fp["label"][::-1], top_fp["support"][::-1], color=PALETTE[0])
axes[0].set_title("Top FP-Growth Itemsets"); axes[0].tick_params(axis="y", labelsize=7)

top_ps = ps_patterns[:15]
ps_labels = [" -> ".join(p) for p, _ in top_ps]
ps_supports = [s for _, s in top_ps]
axes[1].barh(ps_labels[::-1], ps_supports[::-1], color=PALETTE[1])
axes[1].set_title("Top Sequential Patterns"); axes[1].tick_params(axis="y", labelsize=7)

sp_by_class = pd.DataFrame({
    "label": [SESSIONS_TEST[s]["label"] for s in test_session_ids],
    "SP_t": SP_t_test_arr,
})
sns.boxplot(data=sp_by_class, x="SP_t", y="label", ax=axes[2], palette=PALETTE)
axes[2].set_title("SP_t by True Class (Test)")

fig.suptitle("Stage 8 — FP-Growth + PrefixSpan Pattern Mining", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage8_pattern_mining.png", bbox_inches="tight")
plt.show()


## Stages 9-10 — Attack-State Graph & Temporal Consistency Scoring

Builds a 10-node token-transition graph from real training sessions with
EMA-updated edge weights, then scores held-out sessions by how consistent
their observed transitions are with the learned graph (`TC_t`), combined
with a betweenness-centrality graph-weight score (`G_w`).


In [ ]:
from src.nids.attack_graph import (
    build_attack_graph, compute_tc_scores_batch, compute_graph_centrality, compute_gw_batch,
)

G_t = build_attack_graph(SESSIONS_TRAIN, rho=EMA_RHO)

TC_t_val = compute_tc_scores_batch(SESSIONS_VAL, G_t, lam=LAMBDA_DECAY)
TC_t_test = compute_tc_scores_batch(SESSIONS_TEST, G_t, lam=LAMBDA_DECAY)
TC_t_val_arr = np.array([TC_t_val[s] for s in val_session_ids])
TC_t_test_arr = np.array([TC_t_test[s] for s in test_session_ids])

centrality = compute_graph_centrality(G_t)
G_w_val = compute_gw_batch(SESSIONS_VAL, centrality)
G_w_test = compute_gw_batch(SESSIONS_TEST, centrality)
G_w_val_arr = np.array([G_w_val[s] for s in val_session_ids])
G_w_test_arr = np.array([G_w_test[s] for s in test_session_ids])

print(f"Graph nodes: {G_t.number_of_nodes()} | Graph edges: {G_t.number_of_edges()}")
top_edges = sorted(G_t.edges(data=True), key=lambda e: e[2]["weight"], reverse=True)[:10]
print("\nTop-10 transitions by weight:")
for src, dst, data in top_edges:
    print(f"  {src} -> {dst} : {data['weight']:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

pos = nx.spring_layout(G_t, seed=SEED, k=0.9)
edge_weights = [G_t[u][v]["weight"] for u, v in G_t.edges()]
nx.draw_networkx_nodes(G_t, pos, ax=axes[0], node_color=PALETTE[:len(G_t.nodes())], node_size=1400)
nx.draw_networkx_labels(G_t, pos, ax=axes[0], font_size=7)
edges = nx.draw_networkx_edges(
    G_t, pos, ax=axes[0], edge_color=edge_weights, edge_cmap=plt.cm.viridis,
    width=2, arrowsize=15, connectionstyle="arc3,rad=0.08",
)
axes[0].set_title("Attack-State Transition Graph (edge color = weight)")
axes[0].axis("off")

tc_by_class = pd.DataFrame({
    "label": [SESSIONS_TEST[s]["label"] for s in test_session_ids],
    "TC_t": TC_t_test_arr,
})
sns.boxplot(data=tc_by_class, x="TC_t", y="label", ax=axes[1], palette=PALETTE)
axes[1].set_title("TC_t by True Class (Test)")

fig.suptitle("Stages 9-10 — Attack-State Graph & Temporal Consistency", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage9_10_graph.png", bbox_inches="tight")
plt.show()


## Stage 11 — Adaptive Risk Meta-Learner

Re-fuses validation/test evidence with the real `SP_t`/`TC_t` signals,
builds the `(K+4)`-dimensional meta-feature vector
`[R_t, SP_t, TC_t, G_w, dt_inv]`, and trains a logistic-regression
meta-learner that outputs a single calibrated `Risk_t ∈ [0, 1]` per
session, thresholded into BENIGN / SUSPICIOUS / ATTACK tiers.


In [ ]:
from src.nids.risk import (
    build_meta_features, compute_dt_inv, train_meta_learner, predict_risk, classify_sessions, risk_to_tier,
)

R_val_final = fuse(P_A_val_sess, P_B_val, P_C_val_sess, SP_t_val_arr, TC_t_val_arr, *best_weights)
R_test_final = fuse(P_A_test_sess, P_B_test, P_C_test_sess, SP_t_test_arr, TC_t_test_arr, *best_weights)

dt_inv_val = compute_dt_inv(SESSIONS_VAL)
dt_inv_test = compute_dt_inv(SESSIONS_TEST)
dt_inv_val_arr = np.array([dt_inv_val[s] for s in val_session_ids])
dt_inv_test_arr = np.array([dt_inv_test[s] for s in test_session_ids])

X_meta_val = build_meta_features(R_val_final, SP_t_val_arr, TC_t_val_arr, G_w_val_arr, dt_inv_val_arr)
X_meta_test = build_meta_features(R_test_final, SP_t_test_arr, TC_t_test_arr, G_w_test_arr, dt_inv_test_arr)

benign_id = le.transform(["BENIGN"])[0] if "BENIGN" in list(CLASSES) else 0
y_risk_val = (y_val_sess != benign_id).astype(int)
y_risk_test = (y_test_sess != benign_id).astype(int)

meta_lr = train_meta_learner(X_meta_val, y_risk_val)
Risk_t_test = predict_risk(meta_lr, X_meta_test)
alert_tiers = classify_sessions(Risk_t_test)

tier_counts = pd.Series(alert_tiers).value_counts()
print("Alert tier distribution (test):")
print(tier_counts)

risk_by_class = pd.DataFrame({
    "label": [SESSIONS_TEST[s]["label"] for s in test_session_ids],
    "Risk_t": Risk_t_test,
})
print("\nMean risk by true class:")
print(risk_by_class.groupby("label")["Risk_t"].mean().sort_values(ascending=False))


In [ ]:
from sklearn.metrics import roc_curve as _roc_curve, auc as _auc

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

benign_mask = risk_by_class["label"] == "BENIGN"
axes[0].hist(risk_by_class.loc[benign_mask, "Risk_t"], bins=30, alpha=0.6, label="Benign", color=PALETTE[0])
axes[0].hist(risk_by_class.loc[~benign_mask, "Risk_t"], bins=30, alpha=0.6, label="Attack", color=PALETTE[3])
axes[0].axvline(RISK_BENIGN_THRESH, color="gray", linestyle="--")
axes[0].axvline(RISK_SUSPICIOUS_THRESH, color="black", linestyle="--")
axes[0].set_title("Risk Score Distribution"); axes[0].legend()

axes[1].pie(tier_counts.values, labels=tier_counts.index, autopct="%1.1f%%", colors=PALETTE[4:7],
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Alert Tier Distribution")

fpr, tpr, _ = _roc_curve(y_risk_test, Risk_t_test)
axes[2].plot(fpr, tpr, color=PALETTE[5], label=f"AUC={_auc(fpr, tpr):.3f}")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR"); axes[2].legend()
axes[2].set_title("Risk_t ROC (Benign vs Attack)")

fig.suptitle("Stage 11 — Adaptive Risk Meta-Learner", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage11_risk.png", bbox_inches="tight")
plt.show()


## Stage 12 — Streaming Evaluation

Replays the test-session stream through sliding time windows
(`STREAM_WINDOW_SIZE`s, stride `STREAM_STRIDE`s), tracking macro-F1,
per-window inference latency, throughput, and alert rate — the operational
metrics a SOC would monitor for a real-time deployment.


In [ ]:
from src.nids.streaming import make_synthetic_timestamps, evaluate_streaming

session_timestamps = make_synthetic_timestamps(len(test_session_ids))
streaming_results = evaluate_streaming(
    test_session_ids, session_timestamps, y_risk_test,
    R_test_final, SP_t_test_arr, TC_t_test_arr, G_w_test_arr, dt_inv_test_arr,
    meta_lr, window_size=STREAM_WINDOW_SIZE, stride=STREAM_STRIDE,
)

print(f"Windows evaluated : {len(streaming_results)}")
print(f"Mean macro-F1     : {streaming_results['macro_f1'].mean():.4f}")
print(f"Mean latency (ms) : {streaming_results['latency_ms'].mean():.4f}")
print(f"Mean throughput   : {streaming_results['throughput'].mean():.1f} sessions/sec")
streaming_results.head(10)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(streaming_results["window_start"], streaming_results["macro_f1"], color=PALETTE[0], marker="o", ms=3)
axes[0, 0].set_title("Macro-F1 over Time"); axes[0, 0].set_xlabel("Window start (s)")

axes[0, 1].plot(streaming_results["window_start"], streaming_results["latency_ms"], color=PALETTE[1], marker="o", ms=3)
axes[0, 1].set_title("Inference Latency over Time"); axes[0, 1].set_xlabel("Window start (s)"); axes[0, 1].set_ylabel("ms")

axes[1, 0].plot(streaming_results["window_start"], streaming_results["throughput"], color=PALETTE[2], marker="o", ms=3)
axes[1, 0].set_title("Throughput over Time"); axes[1, 0].set_xlabel("Window start (s)"); axes[1, 0].set_ylabel("sessions/sec")

axes[1, 1].plot(streaming_results["window_start"], streaming_results["alert_rate"], color=PALETTE[3], marker="o", ms=3)
axes[1, 1].set_title("Alert Rate over Time"); axes[1, 1].set_xlabel("Window start (s)")

fig.suptitle("Stage 12 — Streaming Evaluation", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage12_streaming.png", bbox_inches="tight")
plt.show()


## Stage 13 — Explainability & Evidence Chain

Produces analyst-facing explanations: TreeSHAP attributions for the RF and
XGBoost branches, attack-graph path evidence for a flagged session's token
sequence, and a consolidated evidence-chain report suitable for SOC
Tier-2 triage.


In [ ]:
### 13a — TreeSHAP (Random Forest)
import shap
from src.nids.explainability import compute_treeshap_rf, compute_treeshap_xgb, extract_graph_path_evidence, build_evidence_report

shap_sample_n = min(SHAP_SAMPLE_SIZE, X_rf_test.shape[0])
X_rf_shap_sample = X_rf_test[:shap_sample_n]
explainer_rf, sv_rf = compute_treeshap_rf(rf, X_rf_shap_sample, RF_FEATURES)

attack_class_idx = int(np.argmax([c != "BENIGN" for c in CLASSES]) if "BENIGN" in CLASSES else 1)
sv_rf_plot = sv_rf[attack_class_idx] if isinstance(sv_rf, list) else sv_rf
plt.figure(figsize=(10, 8))
shap.summary_plot(sv_rf_plot, X_rf_shap_sample, feature_names=RF_FEATURES, show=False)
plt.title(f"RF TreeSHAP — class '{CLASSES[attack_class_idx]}'")
plt.tight_layout()
plt.savefig(FIG_DIR / "stage13a_shap_rf.png", bbox_inches="tight")
plt.show()


In [ ]:
### 13b — TreeSHAP (XGBoost) + side-by-side importance
shap_sample_n_xgb = min(SHAP_SAMPLE_SIZE, X_xgb_test.shape[0])
X_xgb_shap_sample = X_xgb_test[:shap_sample_n_xgb]
explainer_xgb, sv_xgb = compute_treeshap_xgb(xgb_clf, X_xgb_shap_sample, XGB_FEATURES)

def _mean_abs_shap(sv, class_idx):
    vals = sv[class_idx] if isinstance(sv, list) else sv
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[:, :, class_idx]
    return np.abs(vals).mean(axis=0)

# xgb_clf.classes_ / sv_xgb columns are in xgb_clf's LOCAL (present-classes-only)
# order, which can differ from the global class-id space if a rare class was
# fully removed by SMOTE+ENN; map attack_class_idx into that local index.
xgb_local_idx = xgb_clf.local_index_of(attack_class_idx)
if xgb_local_idx < 0:
    xgb_local_idx = 0

rf_importance = pd.Series(_mean_abs_shap(sv_rf, attack_class_idx), index=RF_FEATURES).sort_values(ascending=False).head(15)
xgb_importance = pd.Series(_mean_abs_shap(sv_xgb, xgb_local_idx), index=XGB_FEATURES).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].barh(rf_importance.index[::-1], rf_importance.values[::-1], color=PALETTE[0])
axes[0].set_title("RF — Mean |SHAP| (Top 15)")
axes[1].barh(xgb_importance.index[::-1], xgb_importance.values[::-1], color=PALETTE[1])
axes[1].set_title("XGBoost — Mean |SHAP| (Top 15)")
fig.suptitle("Stage 13b — RF vs XGBoost Feature Importance", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage13_shap_importance.png", bbox_inches="tight")
plt.show()


In [ ]:
### 13c — Graph path evidence for the first flagged ATTACK session
_flagged = [i for i, r in enumerate(Risk_t_test) if r >= RISK_SUSPICIOUS_THRESH]
attack_sess_idx = _flagged[0] if _flagged else int(np.argmax(Risk_t_test))
attack_sess_id = test_session_ids[attack_sess_idx]
attack_tokens = SESSIONS_TEST[attack_sess_id]["tokens"]
path_evidence = extract_graph_path_evidence(attack_tokens, G_t)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

token_idx = [VOCAB.index(t) for t in attack_tokens]
axes[0].step(range(len(token_idx)), token_idx, where="mid", color=PALETTE[3], marker="o")
axes[0].set_yticks(range(len(VOCAB))); axes[0].set_yticklabels(VOCAB, fontsize=8)
axes[0].set_xlabel("Event index"); axes[0].set_title(f"Flagged Session Token Sequence ({attack_sess_id})")

path_nodes = list(dict.fromkeys(attack_tokens))
subG = G_t.subgraph(path_nodes)
pos2 = nx.spring_layout(subG, seed=SEED)
edge_colors = ["red" if e["novel"] else "green" for e in path_evidence]
nx.draw_networkx_nodes(subG, pos2, ax=axes[1], node_color=PALETTE[:len(path_nodes)], node_size=1200)
nx.draw_networkx_labels(subG, pos2, ax=axes[1], font_size=8)
nx.draw_networkx_edges(subG, pos2, ax=axes[1], arrowsize=15)
axes[1].set_title("Path Subgraph (red border = novel transition)")
axes[1].axis("off")

fig.suptitle("Stage 13c — Attack-Graph Path Evidence", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage13c_graph_evidence.png", bbox_inches="tight")
plt.show()


In [ ]:
### 13d — Full analyst evidence-chain report
report = build_evidence_report(
    attack_sess_id, SESSIONS_TEST[attack_sess_id],
    risk_score=Risk_t_test[attack_sess_idx],
    sp_score=SP_t_test_arr[attack_sess_idx],
    tc_score=TC_t_test_arr[attack_sess_idx],
    gw_score=G_w_test_arr[attack_sess_idx],
    shap_values_rf=sv_rf, rf_feature_names=RF_FEATURES,
    path_evidence=path_evidence,
)
print(report)


## Final Summary

In [ ]:
final_results = comparison.copy()
final_results.loc[len(final_results)] = {
    "model": "Fused (Stage 11 Risk_t)", "features": "R_t + SP_t + TC_t + G_w + dt_inv",
    "macro_f1": f1_score(y_risk_test, (Risk_t_test >= RISK_SUSPICIOUS_THRESH).astype(int), average="macro", zero_division=0),
    "accuracy": accuracy_score(y_risk_test, (Risk_t_test >= RISK_SUSPICIOUS_THRESH).astype(int)),
}
print(final_results)

fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(final_results["model"], final_results["macro_f1"], color=PALETTE[: len(final_results)])
ax.set_ylim(0, 1)
ax.set_ylabel("Macro-F1")
ax.set_title("Final Model Comparison — Macro-F1")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "final_results_summary.png", bbox_inches="tight")
plt.show()

print("\n\u2705 All 13 stages complete. Ready for IEEE submission.")
